# Salting for skewed dataset

In [1]:
try:
    spark.stop()
except:
    pass

In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
F = f

In [3]:
# Initialize Spark session
spark = (SparkSession.builder.appName("SparkDataFrameDemo")
         .config("spark.default.parallelism", "4")
         .getOrCreate()
        )

# Partition setting

Although this setting does not seem to have a visible effect on the results
in this example, the number of partitions is used behind the scenes
e.g. in groupBy() clauses for grouping in partitions and aggregation.

Unclear how to check if the partition number is actually respected.

In [4]:
spark.conf.set("spark.sql.shuffle.partitions", 8)

In [5]:
# try to disable the default behavior of "shuffle partitions coalesce" enabled with AQE
spark.conf.set("spark.sql.adaptive.enabled", False)

In [6]:
# THIS IS NOT DOING ANYTHING
# MUST BE SET AT SESSION BUILT TIME
# see spark-rdd-basics.ipynb in section "Demo ETL" little below half
spark.conf.set("spark.default.parallelism", 5)

In [7]:
# create data with 1 million heavily skewed rows
data = (
    *(("A", i) for i in range(900_000)),
    *(("B", i) for i in range(50_000)),
    *(("C", i) for i in range(50_000)),
)
data[:10]

(('A', 0),
 ('A', 1),
 ('A', 2),
 ('A', 3),
 ('A', 4),
 ('A', 5),
 ('A', 6),
 ('A', 7),
 ('A', 8),
 ('A', 9))

In [8]:
df_org = _df = spark.createDataFrame(data, ["key", "value"])

In [9]:
df_org.show(10)
print("Total rows:", df_org.count())

+---+-----+
|key|value|
+---+-----+
|  A|    0|
|  A|    1|
|  A|    2|
|  A|    3|
|  A|    4|
|  A|    5|
|  A|    6|
|  A|    7|
|  A|    8|
|  A|    9|
+---+-----+
only showing top 10 rows

Total rows: 1000000


In [10]:
print("Num  partitions: ", _df.rdd.getNumPartitions())

Num  partitions:  4


In [36]:
(
    df_org
    .withColumn("partition_id", f.spark_partition_id())
    .groupby("partition_id")
    .count()
    .orderBy("partition_id")
    .show()
) 

+------------+------+
|partition_id| count|
+------------+------+
|           0|249856|
|           1|249856|
|           2|249856|
|           3|250432|
+------------+------+



In [11]:
(
    df_org
    .withColumn("partition_id", f.spark_partition_id())
    .groupby("partition_id", "key",)
    .count()
    .orderBy("partition_id", "key")
    .show()
) 

+------------+---+------+
|partition_id|key| count|
+------------+---+------+
|           0|  A|249856|
|           1|  A|249856|
|           2|  A|249856|
|           3|  A|150432|
|           3|  B| 50000|
|           3|  C| 50000|
+------------+---+------+



In [12]:
# manual repartitioning using column "key" makes it worse 
# because data in the column is skewed
(
    df_org
    
    .repartition(6, "key")
    
    .withColumn("partition_id", f.spark_partition_id())
    .groupby("partition_id", "key",)
    .count()
    .orderBy("partition_id", "key")
    .show()
)

+------------+---+------+
|partition_id|key| count|
+------------+---+------+
|           1|  C| 50000|
|           2|  A|900000|
|           3|  B| 50000|
+------------+---+------+



In [54]:
# repartitioning with 6 partitions without column seems ok
# All values of "key" are evenly distributed to partitions
df_org.show(5)

df_org_repartitioned = (
    df_org    
    .repartition(6)
)
df_org_repartitioned.show(5)

+---+-----+
|key|value|
+---+-----+
|  A|    0|
|  A|    1|
|  A|    2|
|  A|    3|
|  A|    4|
+---+-----+
only showing top 5 rows

+---+------+
|key| value|
+---+------+
|  A|157900|
|  A|124128|
|  A|159029|
|  A|155592|
|  A|203713|
+---+------+
only showing top 5 rows



In [55]:
(
    df_org_repartitioned
    .withColumn("partition_id", f.spark_partition_id())
    .groupby("partition_id", "key",)
    .count()
    .orderBy("partition_id", "key")
    .show()
)

+------------+---+------+
|partition_id|key| count|
+------------+---+------+
|           0|  A|150001|
|           0|  B|  8333|
|           0|  C|  8334|
|           1|  A|150000|
|           1|  B|  8333|
|           1|  C|  8334|
|           2|  A|149999|
|           2|  B|  8334|
|           2|  C|  8333|
|           3|  A|149999|
|           3|  B|  8334|
|           3|  C|  8333|
|           4|  A|150000|
|           4|  B|  8333|
|           4|  C|  8333|
|           5|  A|150001|
|           5|  B|  8333|
|           5|  C|  8333|
+------------+---+------+



## In a cluster env calling groupby will force re-shuffle

In [13]:
#print(_df.rdd.toDebugString().decode())

In [14]:
#print(_df.groupby("key").count().rdd.toDebugString().decode())

In [57]:
#_df = df_org
_df = df_org_repartitioned

In [62]:
# call groupby to force re-shuffle
df_grouped = _df.groupby("key").count() # .explain(mode="formatted")

In [59]:
(
    df_grouped
    .withColumn("partition_id", f.spark_partition_id())
    .show()
) 

+---+------+------------+
|key| count|partition_id|
+---+------+------------+
|  C| 50000|           1|
|  B| 50000|           1|
|  A|900000|           2|
+---+------+------------+



In [67]:
result = (
    df_grouped
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy(f.col("partition_id"))
    .count()
) 
result.show()

num_partitions = result.count()
print("Num used partitions: ", num_partitions)
print("Num actual partitions: ", df_grouped.rdd.getNumPartitions())

+------------+-----+
|partition_id|count|
+------------+-----+
|           1|    2|
|           2|    1|
+------------+-----+

Num used partitions:  2
Num actual partitions:  8


## Forcing local repartitioning

In [65]:
df_grouped_repartitioned = df_grouped.repartition(6)

In [66]:
(
    df_grouped_repartitioned
    .withColumn("partition_id", f.spark_partition_id())
    .show()
) 

+---+------+------------+
|key| count|partition_id|
+---+------+------------+
|  C| 50000|           4|
|  B| 50000|           5|
|  A|900000|           5|
+---+------+------------+



In [20]:
result = (
    df_grouped_repartitioned
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy(f.col("partition_id"))
    .count()
) 
result.show(15)

num_partitions = result.count()
print("Num used partitions: ", num_partitions)
print("Num actual partitions: ", df_grouped_repartitioned.rdd.getNumPartitions())

+------------+-----+
|partition_id|count|
+------------+-----+
|           1|    1|
|           3|    1|
|           2|    1|
+------------+-----+

Num used partitions:  3
Num actual partitions:  6


# Adding salt globally splitting each bucket into 10

In [21]:
salt_buckets = 10

_df = df_salted = (
    _df.withColumn(
        "salt",
        (F.rand() * salt_buckets).cast("int")
    )
)
df_salted.show(5)
df_salted.filter(f.col("key") == "B").show(5)
df_salted.filter(f.col("key") == "C").show(5)

+---+-----+----+
|key|value|salt|
+---+-----+----+
|  A|    0|   2|
|  A|    1|   7|
|  A|    2|   1|
|  A|    3|   7|
|  A|    4|   5|
+---+-----+----+
only showing top 5 rows

+---+-----+----+
|key|value|salt|
+---+-----+----+
|  B|    0|   2|
|  B|    1|   9|
|  B|    2|   2|
|  B|    3|   9|
|  B|    4|   7|
+---+-----+----+
only showing top 5 rows

+---+-----+----+
|key|value|salt|
+---+-----+----+
|  C|    0|   7|
|  C|    1|   1|
|  C|    2|   9|
|  C|    3|   5|
|  C|    4|   6|
+---+-----+----+
only showing top 5 rows



In [22]:
_df = df_salted_key = (
    _df.withColumn(
        "salted_key",
        F.concat(F.col("key"), F.lit("_"), F.col("salt"))
    )
)
df_salted_key.take(15)

[Row(key='A', value=0, salt=2, salted_key='A_2'),
 Row(key='A', value=1, salt=7, salted_key='A_7'),
 Row(key='A', value=2, salt=1, salted_key='A_1'),
 Row(key='A', value=3, salt=7, salted_key='A_7'),
 Row(key='A', value=4, salt=5, salted_key='A_5'),
 Row(key='A', value=5, salt=5, salted_key='A_5'),
 Row(key='A', value=6, salt=0, salted_key='A_0'),
 Row(key='A', value=7, salt=2, salted_key='A_2'),
 Row(key='A', value=8, salt=7, salted_key='A_7'),
 Row(key='A', value=9, salt=2, salted_key='A_2'),
 Row(key='A', value=10, salt=9, salted_key='A_9'),
 Row(key='A', value=11, salt=9, salted_key='A_9'),
 Row(key='A', value=12, salt=6, salted_key='A_6'),
 Row(key='A', value=13, salt=7, salted_key='A_7'),
 Row(key='A', value=14, salt=3, salted_key='A_3')]

In [23]:
print("Num partitions: ", df_salted_key.rdd.getNumPartitions())

Num partitions:  4


In [35]:
result = (
    df_salted_key
    .withColumn("partition_id", F.spark_partition_id())
    .groupby("partition_id")
    .count()
    .orderBy("partition_id")
)
result.show()

+------------+------+
|partition_id| count|
+------------+------+
|           0|249856|
|           1|249856|
|           2|249856|
|           3|250432|
+------------+------+



In [74]:
result = (
    df_salted_key
    .withColumn("partition_id", F.spark_partition_id())
    .groupby("partition_id", "salted_key",)
    .count()
    .orderBy("partition_id", "salted_key")
)
result.show()
result.orderBy(f.col("partition_id").desc(), f.col("salted_key").desc()).show(20)

partition_split_salted_key = result

+------------+----------+-----+
|partition_id|salted_key|count|
+------------+----------+-----+
|           0|       A_0|25269|
|           0|       A_1|25086|
|           0|       A_2|25014|
|           0|       A_3|25021|
|           0|       A_4|24622|
|           0|       A_5|25097|
|           0|       A_6|24929|
|           0|       A_7|24976|
|           0|       A_8|24904|
|           0|       A_9|24938|
|           1|       A_0|25242|
|           1|       A_1|24956|
|           1|       A_2|25095|
|           1|       A_3|24783|
|           1|       A_4|25032|
|           1|       A_5|24745|
|           1|       A_6|25102|
|           1|       A_7|25088|
|           1|       A_8|25038|
|           1|       A_9|24775|
+------------+----------+-----+
only showing top 20 rows

+------------+----------+-----+
|partition_id|salted_key|count|
+------------+----------+-----+
|           3|       C_9| 4994|
|           3|       C_8| 4944|
|           3|       C_7| 5007|
|           3|

## In a cluster env calling groupby will force re-shuffle

In [75]:
partition_split_salted_key.show(5)

+------------+----------+-----+
|partition_id|salted_key|count|
+------------+----------+-----+
|           0|       A_0|25269|
|           0|       A_1|25086|
|           0|       A_2|25014|
|           0|       A_3|25021|
|           0|       A_4|24622|
+------------+----------+-----+
only showing top 5 rows



In [79]:
result = (
    partition_split_salted_key
    .groupBy("partition_id")
    .agg(f.count(f.col("salted_key")), f.sum(f.col("count")))
)
result.show()


+------------+-----------------+----------+
|partition_id|count(salted_key)|sum(count)|
+------------+-----------------+----------+
|           1|               10|    249856|
|           3|               30|    250432|
|           2|               10|    249856|
|           0|               10|    249856|
+------------+-----------------+----------+



In [76]:
# call groupby to force re-shuffle
df_grouped_salted = (
    df_salted_key.groupby("salted_key")
    .count()
)

In [77]:
result = (
    df_grouped_salted
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .count()
    .select(f.col("partition_id"), f.col("count").alias("count_of_keys_in_partition"))
    .orderBy("partition_id")
)
result.show()

num_partitions = result.count()
print("Num used partitions: ", num_partitions)
print("Num actual partitions: ", df_grouped.rdd.getNumPartitions())

+------------+--------------------------+
|partition_id|count_of_keys_in_partition|
+------------+--------------------------+
|           0|                         3|
|           1|                         1|
|           2|                         3|
|           3|                         4|
|           4|                         6|
|           5|                         3|
|           6|                         6|
|           7|                         4|
+------------+--------------------------+

Num used partitions:  8
Num actual partitions:  8


In [73]:
result = (
    df_grouped_salted
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id", "salted_key")
    .count()
    .orderBy("partition_id", "salted_key")
)
result.show()

num_partitions = result.count()
print("Num used partition keys: ", num_partitions)
print("Num actual partitions: ", df_grouped.rdd.getNumPartitions())

+------------+----------+-----+
|partition_id|salted_key|count|
+------------+----------+-----+
|           0|       A_3|    1|
|           0|       A_9|    1|
|           0|       C_6|    1|
|           1|       C_2|    1|
|           2|       A_0|    1|
|           2|       A_7|    1|
|           2|       C_8|    1|
|           3|       B_1|    1|
|           3|       B_5|    1|
|           3|       B_9|    1|
|           3|       C_4|    1|
|           4|       A_4|    1|
|           4|       A_8|    1|
|           4|       B_6|    1|
|           4|       B_8|    1|
|           4|       C_3|    1|
|           4|       C_5|    1|
|           5|       A_1|    1|
|           5|       C_1|    1|
|           5|       C_7|    1|
+------------+----------+-----+
only showing top 20 rows

Num used partition keys:  30
Num actual partitions:  8


## Forcing local repartitioning

In [51]:
df_grouped_salted_repartitioned = df_grouped_salted.repartition(
    100,
    "salted_key",
)

In [62]:
result = (
    df_grouped_salted_repartitioned
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy(f.col("partition_id"))
    .count()
    .orderBy(f.col("count").desc())
) 
result.show(15)

num_partitions = result.count()
print("Num used partitions: ", num_partitions)
print("Num actual partitions: ", df_grouped_salted_repartitioned.rdd.getNumPartitions())


+------------+-----+
|partition_id|count|
+------------+-----+
|           5|    2|
|           6|    2|
|          36|    2|
|          71|    2|
|          10|    1|
|          12|    1|
|          19|    1|
|          28|    1|
|          35|    1|
|          40|    1|
|          43|    1|
|          46|    1|
|          47|    1|
|          51|    1|
|          52|    1|
+------------+-----+
only showing top 15 rows

Num used partitions:  26
Num actual partitions:  100


In [37]:
(
    df_grouped_salted_repartitioned
    .withColumn("partition_id", f.spark_partition_id())
    .orderBy(f.col("salted_key"))
    .show(30)
) 

+----------+-----+------------+
|salted_key|count|partition_id|
+----------+-----+------------+
|       A_0|90192|           6|
|       A_1|89851|          73|
|       A_2|90319|          90|
|       A_3|90023|          36|
|       A_4|89776|          12|
|       A_5|90126|          82|
|       A_6|90186|          35|
|       A_7|89659|          66|
|       A_8|90063|          40|
|       A_9|89805|          84|
|       B_0| 4853|          71|
|       B_1| 5048|          75|
|       B_2| 5069|          71|
|       B_3| 4914|          46|
|       B_4| 5042|          54|
|       B_5| 4942|          19|
|       B_6| 5030|          28|
|       B_7| 4938|          62|
|       B_8| 5055|          68|
|       B_9| 5109|          43|
|       C_0| 5070|          51|
|       C_1| 4987|           5|
|       C_2| 4985|           5|
|       C_3| 4960|          52|
|       C_4| 5083|          47|
|       C_5| 5015|          80|
|       C_6| 5004|          36|
|       C_7| 4886|          97|
|       

In [38]:
(
    df_grouped_salted_repartitioned
    .withColumn("partition_id", f.spark_partition_id())
    .orderBy(f.col("partition_id"))
    .show(30)
) 

+----------+-----+------------+
|salted_key|count|partition_id|
+----------+-----+------------+
|       C_2| 4985|           5|
|       C_1| 4987|           5|
|       A_0|90192|           6|
|       C_9| 4945|           6|
|       C_8| 5065|          10|
|       A_4|89776|          12|
|       B_5| 4942|          19|
|       B_6| 5030|          28|
|       A_6|90186|          35|
|       A_3|90023|          36|
|       C_6| 5004|          36|
|       A_8|90063|          40|
|       B_9| 5109|          43|
|       B_3| 4914|          46|
|       C_4| 5083|          47|
|       C_0| 5070|          51|
|       C_3| 4960|          52|
|       B_4| 5042|          54|
|       B_7| 4938|          62|
|       A_7|89659|          66|
|       B_8| 5055|          68|
|       B_2| 5069|          71|
|       B_0| 4853|          71|
|       A_1|89851|          73|
|       B_1| 5048|          75|
|       C_5| 5015|          80|
|       A_5|90126|          82|
|       A_9|89805|          84|
|       

# Adding salt only for large bucket "A"

In [20]:
salt_buckets = 10

_df = df_salted = (
    _df.withColumn(
        "salt",
        f.when(f.col("key") == "A", 
            (F.rand() * salt_buckets).cast("int")
        ).otherwise(0)
    )
)
df_salted.show(5)
df_salted.filter(f.col("key") == "B").show(5)
df_salted.filter(f.col("key") == "C").show(5)

+---+-----+----+----------+
|key|value|salt|salted_key|
+---+-----+----+----------+
|  A|    0|   0|       A_7|
|  A|    1|   8|       A_9|
|  A|    2|   1|       A_0|
|  A|    3|   9|       A_3|
|  A|    4|   1|       A_0|
+---+-----+----+----------+
only showing top 5 rows

+---+-----+----+----------+
|key|value|salt|salted_key|
+---+-----+----+----------+
|  B|    0|   0|       B_7|
|  B|    1|   0|       B_5|
|  B|    2|   0|       B_1|
|  B|    3|   0|       B_7|
|  B|    4|   0|       B_4|
+---+-----+----+----------+
only showing top 5 rows

+---+-----+----+----------+
|key|value|salt|salted_key|
+---+-----+----+----------+
|  C|    0|   0|       C_3|
|  C|    1|   0|       C_5|
|  C|    2|   0|       C_5|
|  C|    3|   0|       C_8|
|  C|    4|   0|       C_9|
+---+-----+----+----------+
only showing top 5 rows



In [21]:
_df = df_salted_key = (
    _df.withColumn(
        "salted_key",
        F.concat(F.col("key"), F.lit("_"), F.col("salt"))
    )
)
df_salted_key.take(15)

[Row(key='A', value=0, salt=0, salted_key='A_0'),
 Row(key='A', value=1, salt=8, salted_key='A_8'),
 Row(key='A', value=2, salt=1, salted_key='A_1'),
 Row(key='A', value=3, salt=9, salted_key='A_9'),
 Row(key='A', value=4, salt=1, salted_key='A_1'),
 Row(key='A', value=5, salt=6, salted_key='A_6'),
 Row(key='A', value=6, salt=7, salted_key='A_7'),
 Row(key='A', value=7, salt=9, salted_key='A_9'),
 Row(key='A', value=8, salt=9, salted_key='A_9'),
 Row(key='A', value=9, salt=4, salted_key='A_4'),
 Row(key='A', value=10, salt=3, salted_key='A_3'),
 Row(key='A', value=11, salt=4, salted_key='A_4'),
 Row(key='A', value=12, salt=0, salted_key='A_0'),
 Row(key='A', value=13, salt=7, salted_key='A_7'),
 Row(key='A', value=14, salt=8, salted_key='A_8')]

## In a cluster env calling groupby will force re-shuffle

In [22]:
# call groupby to force re-shuffle
df_grouped_salted = df_salted_key.groupby("salted_key").count()

In [23]:
(
    df_grouped_salted
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .count()
    .show()
)

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|   12|
+------------+-----+



In [24]:
(
    df_salted_key
    .withColumn("partition_id", F.spark_partition_id())
    #.groupBy("partition_id")
    #.count()
    .show()
)

+---+-----+----+----------+------------+
|key|value|salt|salted_key|partition_id|
+---+-----+----+----------+------------+
|  A|    0|   0|       A_0|           0|
|  A|    1|   8|       A_8|           0|
|  A|    2|   1|       A_1|           0|
|  A|    3|   9|       A_9|           0|
|  A|    4|   1|       A_1|           0|
|  A|    5|   6|       A_6|           0|
|  A|    6|   7|       A_7|           0|
|  A|    7|   9|       A_9|           0|
|  A|    8|   9|       A_9|           0|
|  A|    9|   4|       A_4|           0|
|  A|   10|   3|       A_3|           0|
|  A|   11|   4|       A_4|           0|
|  A|   12|   0|       A_0|           0|
|  A|   13|   7|       A_7|           0|
|  A|   14|   8|       A_8|           0|
|  A|   15|   3|       A_3|           0|
|  A|   16|   6|       A_6|           0|
|  A|   17|   4|       A_4|           0|
|  A|   18|   8|       A_8|           0|
|  A|   19|   6|       A_6|           0|
+---+-----+----+----------+------------+
only showing top

## Forcing local repartitioning

In [25]:
df_grouped_salted_repartitioned = df_grouped_salted.repartition(
    100,
    "salted_key",
)

In [26]:
result = (
    df_grouped_salted_repartitioned
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy(f.col("partition_id"))
    .count()
    .orderBy(f.col("count").desc())
) 
result.show(15)

num_partitions = result.count()
print("Num partitions: ", num_partitions)

+------------+-----+
|partition_id|count|
+------------+-----+
|           6|    1|
|          12|    1|
|          35|    1|
|          36|    1|
|          40|    1|
|          51|    1|
|          66|    1|
|          71|    1|
|          73|    1|
|          82|    1|
|          84|    1|
|          90|    1|
+------------+-----+

Num partitions:  12


In [27]:
(
    df_grouped_salted_repartitioned
    .withColumn("partition_id", f.spark_partition_id())
    .orderBy(f.col("salted_key"))
    .show(30)
) 

+----------+-----+------------+
|salted_key|count|partition_id|
+----------+-----+------------+
|       A_0|89984|           6|
|       A_1|89590|          73|
|       A_2|90273|          90|
|       A_3|89881|          36|
|       A_4|89954|          12|
|       A_5|90210|          82|
|       A_6|89653|          35|
|       A_7|89896|          66|
|       A_8|90169|          40|
|       A_9|90390|          84|
|       B_0|50000|          71|
|       C_0|50000|          51|
+----------+-----+------------+



In [28]:
(
    df_grouped_salted_repartitioned
    .withColumn("partition_id", f.spark_partition_id())
    .orderBy(f.col("partition_id"))
    .show(20)
) 

+----------+-----+------------+
|salted_key|count|partition_id|
+----------+-----+------------+
|       A_0|89984|           6|
|       A_4|89954|          12|
|       A_6|89653|          35|
|       A_3|89881|          36|
|       A_8|90169|          40|
|       C_0|50000|          51|
|       A_7|89896|          66|
|       B_0|50000|          71|
|       A_1|89590|          73|
|       A_5|90210|          82|
|       A_9|90390|          84|
|       A_2|90273|          90|
+----------+-----+------------+

